# LeetCode 1503 — Last Moment Before All Ants Fall Out of a Plank bằng Python

Link đề: https://leetcode.com/problems/last-moment-before-all-ants-fall-out-of-a-plank/ · Độ khó: **Medium** · Chủ đề: Array, Brainteaser, Simulation

Skeleton LeetCode:

```python
class Solution:
    def getLastMoment(self, n: int, left: list[int], right: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô phỏng chính xác chuyển động và va chạm bằng cách **nhân đôi toạ độ** (bước nửa giây) để mọi cuộc gặp xảy ra tại điểm nguyên.
2. Nắm "mẹo não" kinh điển: hai con kiến va chạm rồi quay đầu **tương đương** với hai con kiến **đi xuyên qua nhau** (chỉ đổi nhãn) ⇒ va chạm không ảnh hưởng tới thời điểm cuối cùng.
3. Rút ra công thức `max(max(left), n − min(right))` và xử lý mảng rỗng (`default=0`).
4. Biết thêm tính chất **giữ thứ tự** của các con kiến để tính thời điểm rơi của **từng** con kiến thật.
5. Kiểm chứng "mẹo" bằng mô phỏng trên hàng nghìn bộ test ngẫu nhiên.

## 2. Đề bài & ví dụ

Có một tấm ván dài `n` đơn vị (các vị trí từ `0` tới `n`). Một số con kiến đang đi trên ván với vận tốc 1 đơn vị/giây; mảng `left` chứa vị trí các con đi **sang trái**, mảng `right` chứa vị trí các con đi **sang phải**.

- Khi hai con kiến đi ngược chiều **gặp nhau**, cả hai **đổi hướng** ngay lập tức (không mất thời gian).
- Khi một con kiến tới một đầu ván ở thời điểm `t`, nó **rơi khỏi ván ngay**.

Trả về thời điểm con kiến (hoặc các con) **cuối cùng** rơi khỏi ván.

**Ví dụ 1:** `n = 4, left = [4,3], right = [0,1]` → `4`. (Kiến A ở 0 và B ở 1 đi phải; C ở 3 và D ở 4 đi trái. Sau `t = 4` không còn con nào trên ván.)

**Ví dụ 2:** `n = 7, left = [], right = [0,1,2,3,4,5,6,7]` → `7` (con ở 0 cần 7 giây).

**Ví dụ 3:** `n = 7, left = [0,1,2,3,4,5,6,7], right = []` → `7` (con ở 7 cần 7 giây).

**Ràng buộc:** `1 <= n <= 10^4`; `0 <= len(left), len(right) <= n + 1`; `0 <= left[i], right[i] <= n`; `1 <= len(left) + len(right) <= n + 1`; mọi vị trí phân biệt, mỗi giá trị chỉ nằm ở một trong hai mảng.

In [1]:
import random
import time
from typing import List

examples = [((4, [4, 3], [0, 1]), 4), ((7, [], [0, 1, 2, 3, 4, 5, 6, 7]), 7), ((7, [0, 1, 2, 3, 4, 5, 6, 7], []), 7)]
for (n, left, right), want in examples:
    print(f"n={n}, left={left}, right={right} -> {want}")

n=4, left=[4, 3], right=[0, 1] -> 4
n=7, left=[], right=[0, 1, 2, 3, 4, 5, 6, 7] -> 7
n=7, left=[0, 1, 2, 3, 4, 5, 6, 7], right=[] -> 7


## 3. Kiến thức nền

### 3.1 `max` / `min` với mảng rỗng

`max([])` báo `ValueError`. Python cho tham số `default`: `max([], default=0)` trả `0`. Bài này `left` hoặc `right` có thể rỗng.

In [2]:
print(max([3, 9, 1], default=0), max([], default=0), min([], default=10))
try:
    max([])
except ValueError as e:
    print("max([]) ->", e)

9 0 10
max([]) -> max() iterable argument is empty


### 3.2 Nhân đôi toạ độ để làm việc với số nguyên

Hai con kiến ở vị trí nguyên `3` và `4` đi ngược chiều sẽ gặp nhau ở `3.5` sau `0.5` giây. Nhân mọi vị trí và mọi mốc thời gian với 2: vị trí `6` và `8`, gặp ở `7` sau **1 bước** (1 bước = 0,5 giây). Mọi thứ thành số nguyên, không lo sai số số thực.

In [3]:
a, b = 3, 4
A, B = 2 * a, 2 * b
step = 0
while A != B:
    A += 1; B -= 1; step += 1
print(f"gặp tại toạ độ nhân đôi {A} (= {A / 2}) sau {step} bước = {step / 2} giây")

gặp tại toạ độ nhân đôi 7 (= 3.5) sau 1 bước = 0.5 giây


### 3.3 Danh sách các list nhỏ làm "bản ghi"

Mỗi con kiến là một list `[vị_trí_x2, hướng, nhãn]`, hướng `+1` (phải) hoặc `-1` (trái). Sửa trực tiếp phần tử list để cập nhật trạng thái.

In [4]:
ants = [[0, +1, "A"], [8, -1, "D"]]
for a in ants:
    a[0] += a[1]
print(ants)

[[1, 1, 'A'], [7, -1, 'D']]


### 3.4 Gom nhóm theo khoá bằng `dict.setdefault`

Sau mỗi bước, gom các con kiến theo vị trí: vị trí nào có 2 con ⇒ chúng vừa gặp nhau.

In [5]:
ants = [[3, 1, "A"], [3, -1, "B"], [7, 1, "C"]]
by_pos = {}
for a in ants:
    by_pos.setdefault(a[0], []).append(a[2])
print(by_pos)

{3: ['A', 'B'], 7: ['C']}


## 4. Ý tưởng nền tảng / trực giác

Quan sát **Ví dụ 1** (`n = 4`; A ở 0 →, B ở 1 →, C ở 3 ←, D ở 4 ←):

- `t = 1`: B và C gặp nhau ở 2 ⇒ quay đầu (B ←, C →). A ở 1, D ở 3.
- `t = 1.5`: A (→) gặp B (←) ở 1.5 ⇒ quay đầu; C (→) gặp D (←) ở 2.5 ⇒ quay đầu...
- Rối rắm! Nhưng hãy **xoá nhãn** đi và chỉ nhìn tập vị trí "có kiến đi về hướng nào": khi hai con gặp nhau và quay đầu, ngay sau đó vẫn có **một con đi trái và một con đi phải** từ điểm đó — y hệt như nếu hai con **đi xuyên qua nhau**.

Vậy về mặt "bức tranh", các con kiến như những **bóng ma** đi thẳng không bao giờ va chạm:
- bóng ma đi trái từ vị trí `p` rơi ở thời điểm `p`;
- bóng ma đi phải từ vị trí `p` rơi ở thời điểm `n − p`.

Thời điểm cuối cùng còn kiến trên ván = thời điểm bóng ma cuối cùng rơi = `max(max(left), max(n − p for p in right))`.

Ví dụ 1: `max(4, 3, 4 − 0, 4 − 1) = 4` ✔.

In [6]:
for (n, left, right), want in examples:
    ghosts = [p for p in left] + [n - p for p in right]
    print(f"n={n}: thời điểm rơi của các 'bóng ma' = {ghosts} -> max = {max(ghosts)} (mong đợi {want})")

n=4: thời điểm rơi của các 'bóng ma' = [4, 3, 4, 3] -> max = 4 (mong đợi 4)
n=7: thời điểm rơi của các 'bóng ma' = [7, 6, 5, 4, 3, 2, 1, 0] -> max = 7 (mong đợi 7)
n=7: thời điểm rơi của các 'bóng ma' = [0, 1, 2, 3, 4, 5, 6, 7] -> max = 7 (mong đợi 7)


## 5. Cách 1 — Brute Force / baseline: mô phỏng va chạm từng nửa giây

**Ý tưởng:** nhân đôi toạ độ (ván dài `2n`). Mỗi bước (0,5 giây):
1. Con kiến nào đang ở đầu ván và hướng ra ngoài (`0` và đi trái, hoặc `2n` và đi phải) ⇒ rơi tại thời điểm hiện tại.
2. Mọi con còn lại dịch 1 đơn vị theo hướng.
3. Hai con cùng vị trí ⇒ đổi hướng cả hai.

Vì vị trí ban đầu đều chẵn và mọi con cùng di chuyển mỗi bước, tất cả luôn **cùng tính chẵn lẻ** ⇒ hai con đi ngược chiều luôn gặp nhau **đúng tại** một điểm nguyên (không bao giờ "nhảy qua" nhau giữa hai bước).

In [7]:
def simulate(n, left, right, record=False):
    ants = [[2 * p, -1, ("L", p)] for p in left] + [[2 * p, 1, ("R", p)] for p in right]
    fall = {}                      # nhãn ban đầu -> thời điểm rơi (đơn vị nửa giây)
    collisions = 0
    t = 0
    log = []
    while ants:
        keep = []
        for a in ants:
            if (a[0] == 0 and a[1] == -1) or (a[0] == 2 * n and a[1] == 1):
                fall[a[2]] = t
            else:
                keep.append(a)
        ants = keep
        if record:
            log.append((t, sorted((a[0], a[1], a[2]) for a in ants)))
        if not ants:
            break
        for a in ants:
            a[0] += a[1]
        t += 1
        by_pos = {}
        for a in ants:
            by_pos.setdefault(a[0], []).append(a)
        for grp in by_pos.values():
            if len(grp) == 2:      # hai con ngược chiều vừa gặp nhau
                grp[0][1] *= -1
                grp[1][1] *= -1
                collisions += 1
    last = max(fall.values())
    assert last % 2 == 0           # luôn là số giây nguyên
    return last // 2, fall, collisions, log

def last_moment_sim(n, left, right):
    return simulate(n, left, right)[0]

for (n, left, right), want in examples:
    print(last_moment_sim(n, left, right), want)

4 4
7 7
7 7


**Độ phức tạp:** số bước ≤ `2n`, mỗi bước xử lý mọi con kiến `O(A)` (`A = len(left) + len(right) ≤ n + 1`) ⇒ **`O(n · A)`**, tệ nhất `~2·10^8` với `n = 10^4` — quá chậm cho Python.

**Nút thắt:** ta mô phỏng từng va chạm, trong khi (mục 4) va chạm **không làm thay đổi** bức tranh tổng thể. Nếu chứng minh được điều đó thì chỉ cần một phép `max`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (đổi nhãn):** khi kiến X (đi phải) và Y (đi trái) gặp nhau tại điểm `q` ở thời điểm `t`, sau đó X đi trái, Y đi phải. Nếu ta **đổi nhãn** X ↔ Y ngay lúc gặp, thì "X" tiếp tục đi phải, "Y" tiếp tục đi trái — như thể đi xuyên qua nhau. **Tập hợp** (vị trí, hướng) của tất cả kiến tại mọi thời điểm là như nhau ở hai cách nhìn.

**Hệ quả:** thời điểm ván trở nên trống (con cuối cùng rơi) trong thực tế bằng thời điểm đó trong thế giới "đi xuyên qua". Trong thế giới đi xuyên qua, mỗi con đi thẳng một mạch:
- đi trái từ `p` ⇒ rơi ở `p`;
- đi phải từ `p` ⇒ rơi ở `n − p`.

**Quy tắc:** đáp án = `max( max(left), max(n − p) với p ∈ right )` = `max( max(left, default 0), n − min(right, default n) )`.

**Invariant bổ sung (giữ thứ tự):** các con kiến thật **không bao giờ vượt qua nhau** (gặp nhau là quay đầu), nên thứ tự từ trái sang phải được giữ nguyên ⇒ dùng để tính thời điểm rơi của **từng** con thật (Cách 4).

## 7. Các cách tiếp theo

### Cách 2 — Bóng ma, viết bằng vòng lặp tường minh

Dễ đọc cho người mới: duyệt từng con, tính thời điểm "bóng ma" của nó rơi, giữ max.

In [8]:
def last_moment_loop(n, left, right):
    ans = 0
    for p in left:
        ans = max(ans, p)          # đi trái: tới 0 mất p giây
    for p in right:
        ans = max(ans, n - p)      # đi phải: tới n mất n - p giây
    return ans

class SolutionLoop:
    def getLastMoment(self, n: int, left: list[int], right: list[int]) -> int:
        return last_moment_loop(n, left, right)

for (n, left, right), want in examples:
    print(last_moment_loop(n, left, right), want)

4 4
7 7
7 7


**Độ phức tạp:** `O(A)` thời gian, `O(1)` bộ nhớ.

### Cách 3 — Công thức một dòng với `default`

Chỉ cần con đi trái **xa nhất bên phải** (`max(left)`) và con đi phải **xa nhất bên trái** (`min(right)`).

In [9]:
def last_moment_formula(n, left, right):
    return max(max(left, default=0), n - min(right, default=n))

class SolutionFormula:
    def getLastMoment(self, n: int, left: list[int], right: list[int]) -> int:
        return max(max(left, default=0), n - min(right, default=n))

for (n, left, right), want in examples:
    print(last_moment_formula(n, left, right), SolutionFormula().getLastMoment(n, left, right), want)

4 4 4
7 7 7
7 7 7


`min(right, default=n)` ⇒ `n − n = 0` khi `right` rỗng; `max(left, default=0)` ⇒ `0` khi `left` rỗng. **Độ phức tạp:** `O(A)`, `O(1)`.

### Cách 4 — Thời điểm rơi của từng con kiến thật (giữ thứ tự)

Không chỉ đáp án, ta tính được **mỗi con** rơi lúc nào và ở đầu nào:
1. Có đúng `L = len(left)` bóng ma rơi ở đầu trái, tại các thời điểm `sorted(left)`. Vì kiến thật giữ thứ tự và đầu trái chỉ "nhả" con ngoài cùng bên trái trước, **`L` con thật bên trái nhất** rơi ở đầu trái, con thứ `i` (từ trái) rơi ở thời điểm `sorted(left)[i]`.
2. Tương tự, các con còn lại rơi ở đầu phải; con **ngoài cùng bên phải** rơi sớm nhất, ở thời điểm nhỏ nhất trong `sorted(n − p for p in right)`.

Đáp án = max của mọi thời điểm. Cách này còn được dùng để **kiểm tra** mô phỏng theo từng nhãn.

In [10]:
def fall_times(n, left, right):
    labeled = sorted([(p, "L") for p in left] + [(p, "R") for p in right])
    L = len(left)
    left_times = sorted(left)
    right_times = sorted(n - p for p in right)
    res = {}
    for i in range(L):                                   # L con trái nhất rơi đầu trái
        p, d = labeled[i]
        res[(d, p)] = left_times[i]
    for k, i in enumerate(range(len(labeled) - 1, L - 1, -1)):   # từ phải vào, rơi đầu phải
        p, d = labeled[i]
        res[(d, p)] = right_times[k]
    return res

def last_moment_order(n, left, right):
    return max(fall_times(n, left, right).values())

class SolutionOrder:
    def getLastMoment(self, n: int, left: list[int], right: list[int]) -> int:
        return last_moment_order(n, left, right)

print(fall_times(4, [4, 3], [0, 1]))
for (n, left, right), want in examples:
    print(last_moment_order(n, left, right), want)

{('R', 0): 3, ('R', 1): 4, ('L', 4): 3, ('L', 3): 4}
4 4
7 7
7 7


**Độ phức tạp:** `O(A log A)` (sắp xếp), `O(A)` bộ nhớ. Không cần cho bài gốc, nhưng giúp hiểu sâu hơn và kiểm chứng được mô phỏng **theo từng con**.

## 8. TRACE từng bước

Trace mô phỏng (Cách 1) trên **Ví dụ 1**. Toạ độ nhân đôi (ván `0..8`), thời gian theo nửa giây. Nhãn: A=(R,0), B=(R,1), C=(L,3), D=(L,4).

| t (giây) | A | B | C | D | sự kiện |
|---|---|---|---|---|---|
| 0 | 0 → | 1 → | 3 ← | 4 ← | |
| 0.5 | 0.5 → | 1.5 → | 2.5 ← | 3.5 ← | |
| 1 | 1 → | 2 ← | 2 → | 3 ← | B, C gặp ở 2 ⇒ đổi hướng |
| 1.5 | 1.5 ← | 1.5 → | 2.5 ← | 2.5 → | A gặp B ở 1.5; C gặp D ở 2.5 |
| 2 | 1 ← | 2 ← | 2 → | 3 → | B, C gặp lại ở 2 |
| 2.5 | 0.5 ← | 1.5 ← | 2.5 → | 3.5 → | |
| 3 | rơi trái | 1 ← | 3 → | rơi phải | A, D rơi |
| 4 | | rơi trái | rơi phải | | B, C rơi ⇒ **đáp án 4** (4 lần va chạm) |

(Vị trí ghi theo đơn vị gốc; trong code là toạ độ nhân đôi.) Chạy thật để xem đầy đủ:

In [11]:
def trace_sim(n, left, right):
    ans, fall, col, log = simulate(n, left, right, record=True)
    names = {}
    for i, lab in enumerate(sorted([("L", p) for p in left] + [("R", p) for p in right], key=lambda x: x[1])):
        names[lab] = "ABCDEFGHIJ"[i] if i < 10 else str(lab)
    for t, ants in log:
        desc = ", ".join(f"{names[lab]}@{pos / 2:g}{'→' if d > 0 else '←'}" for pos, d, lab in ants)
        fell = [names[lab] for lab, ft in fall.items() if ft == t]
        print(f"t={t / 2:4}s | trên ván: [{desc}]" + (f" | rơi: {fell}" if fell else ""))
    print(f"số va chạm: {col} | đáp án: {ans}")
    return ans

trace_sim(4, [4, 3], [0, 1])

t= 0.0s | trên ván: [A@0→, B@1→, C@3←, D@4←]
t= 0.5s | trên ván: [A@0.5→, B@1.5→, C@2.5←, D@3.5←]
t= 1.0s | trên ván: [A@1→, B@2←, C@2→, D@3←]
t= 1.5s | trên ván: [A@1.5←, B@1.5→, C@2.5←, D@2.5→]
t= 2.0s | trên ván: [A@1←, B@2←, C@2→, D@3→]
t= 2.5s | trên ván: [A@0.5←, B@1.5←, C@2.5→, D@3.5→]
t= 3.0s | trên ván: [B@1←, C@3→] | rơi: ['D', 'A']
t= 3.5s | trên ván: [B@0.5←, C@3.5→]
t= 4.0s | trên ván: [] | rơi: ['C', 'B']
số va chạm: 4 | đáp án: 4


4

So với thế giới "bóng ma": A' (đi phải từ 0) rơi ở 4, B' rơi ở 3, C' (đi trái từ 3) rơi ở 3, D' rơi ở 4. Tập thời điểm rơi `{3, 3, 4, 4}` — **giống hệt** mô phỏng thật, chỉ khác "con nào" rơi. Kiểm tra bằng `fall_times`:

In [12]:
_, fall, _, _ = simulate(4, [4, 3], [0, 1])
print("mô phỏng (theo nhãn, giây):", {k: v / 2 for k, v in sorted(fall.items())})
print("Cách 4 (giữ thứ tự)       :", dict(sorted(fall_times(4, [4, 3], [0, 1]).items())))
print("bóng ma (multiset)        :", sorted([4 - 0, 4 - 1, 3, 4]))

mô phỏng (theo nhãn, giây): {('L', 3): 4.0, ('L', 4): 3.0, ('R', 0): 3.0, ('R', 1): 4.0}
Cách 4 (giữ thứ tự)       : {('L', 3): 4, ('L', 4): 3, ('R', 0): 3, ('R', 1): 4}
bóng ma (multiset)        : [3, 3, 4, 4]


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def getLastMoment(self, n: int, left: list[int], right: list[int]) -> int:
        # Va chạm = đi xuyên qua nhau (đổi nhãn) => mỗi con coi như đi thẳng.
        last_left = max(left, default=0)          # đi trái từ p: rơi sau p giây
        last_right = n - min(right, default=n)    # đi phải từ p: rơi sau n - p giây
        return max(last_left, last_right)

for (n, left, right), want in examples:
    assert Solution().getLastMoment(n, left, right) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

Gọi `S(t)` là **đa tập** các cặp (vị trí, hướng) của những con kiến còn trên ván tại thời điểm `t`, trong thế giới thật; `G(t)` là đa tập tương ứng trong thế giới bóng ma (kiến đi xuyên qua nhau).

**Khẳng định:** `S(t) = G(t)` với mọi `t ≥ 0`.
- `t = 0`: hai thế giới xuất phát giống nhau.
- Giữa hai lần gặp nhau, mọi con đi thẳng ở cả hai thế giới ⇒ hai đa tập biến đổi y như nhau.
- Tại một lần gặp nhau ở điểm `q`: thế giới thật có `(q, →)` và `(q, ←)` đổi thành `(q, ←)` và `(q, →)` — đa tập **không đổi**. Thế giới bóng ma cũng có đúng hai cặp đó. Rơi khỏi ván chỉ phụ thuộc (vị trí, hướng) ⇒ cũng khớp.

Vậy thời điểm cuối cùng ván còn kiến là như nhau ở hai thế giới. Trong thế giới bóng ma, con đi trái từ `p` rơi ở `p`, con đi phải từ `p` rơi ở `n − p` ⇒ đáp án là max của các giá trị đó. Max của `{p : p ∈ left}` là `max(left)`; max của `{n − p : p ∈ right}` là `n − min(right)`.

Bộ test mục 12 xác nhận điều này bằng mô phỏng thật trên hàng nghìn input.

## 11. Phân tích Time / Space

Gọi `A = len(left) + len(right)` (`≤ n + 1 ≤ 10^4 + 1`).

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng nửa giây | `O(n · A)` (tới `~2·10^8`) | `O(A)` |
| 2. Vòng lặp bóng ma | `O(A)` | `O(1)` |
| 3. Công thức `max/min` | `O(A)` | `O(1)` |
| 4. Giữ thứ tự (từng con) | `O(A log A)` | `O(A)` |

Không có khác biệt average/worst cho các cách 2–4.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [14]:
ALL = {
    "sim": last_moment_sim,
    "loop": last_moment_loop,
    "formula": last_moment_formula,
    "order": last_moment_order,
    "sub_loop": lambda n, l, r: SolutionLoop().getLastMoment(n, l, r),
    "sub_formula": lambda n, l, r: SolutionFormula().getLastMoment(n, l, r),
    "sub_order": lambda n, l, r: SolutionOrder().getLastMoment(n, l, r),
    "submit": lambda n, l, r: Solution().getLastMoment(n, l, r),
}
edge = [
    ((1, [0], []), 0),            # rơi ngay ở t = 0
    ((1, [], [1]), 0),
    ((1, [1], [0]), 1),           # hai con đối mặt, gặp ở giữa
    ((5, [5], []), 5),
    ((5, [], [0]), 5),
    ((10, [0], [10]), 0),         # cả hai đang ở đầu và hướng ra ngoài
    ((10, [3, 9], [2, 7]), 9),
    ((20, [4, 7, 15], [9, 3, 13, 10]), 17),
    ((6, [6], [0]), 6),
]
for (n, l, r), want in edge:
    assert last_moment_sim(n, l, r) == want, (n, l, r, last_moment_sim(n, l, r))
for name, f in ALL.items():
    for (n, l, r), want in examples + edge:
        assert f(n, l, r) == want, (name, n, l, r)
print("Mọi cách đúng với", len(examples + edge), "test (edge đã kiểm chứng bằng mô phỏng)")

Mọi cách đúng với 12 test (edge đã kiểm chứng bằng mô phỏng)


### 12.2 Ngẫu nhiên: đối chiếu mô phỏng thật (seed cố định)

Chọn ngẫu nhiên một tập vị trí phân biệt trong `[0, n]` rồi chia ngẫu nhiên vào `left`/`right`. Kiểm tra cả **đáp án** lẫn **thời điểm rơi của từng con** (mô phỏng theo nhãn so với Cách 4).

In [15]:
rng = random.Random(1503)
for _ in range(1500):
    n = rng.randint(1, 30)
    pos = rng.sample(range(n + 1), rng.randint(1, n + 1))
    left = [p for p in pos if rng.random() < 0.5]
    right = [p for p in pos if p not in left]
    if not left and not right:
        continue
    ans, fall, _, _ = simulate(n, left, right)
    per_ant = {k: v // 2 for k, v in fall.items()}
    assert per_ant == fall_times(n, left, right), (n, left, right)
    for name, f in ALL.items():
        assert f(n, left, right) == ans, (name, n, left, right)
print("1500 test ngẫu nhiên: mọi cách khớp mô phỏng; thời điểm rơi từng con khớp Cách 4")

1500 test ngẫu nhiên: mọi cách khớp mô phỏng; thời điểm rơi từng con khớp Cách 4


### 12.3 Giới hạn `n = 10^4`, `A = n + 1`

Mô phỏng ở kích thước này quá chậm, nên chỉ chạy các cách `O(A)`/`O(A log A)`; chạy mô phỏng ở `n = 300` để so thời gian.

In [16]:
n = 10**4
pos = list(range(n + 1)); rng.shuffle(pos)
left_big, right_big = pos[: n // 2], pos[n // 2:]
for name in ["loop", "formula", "order", "submit"]:
    t0 = time.perf_counter(); r = ALL[name](n, left_big, right_big); dt = time.perf_counter() - t0
    print(f"n=1e4 | {name:8}: {r}  ({dt*1000:.2f} ms)")
n2 = 300
pos2 = list(range(n2 + 1)); rng.shuffle(pos2)
t0 = time.perf_counter(); r = last_moment_sim(n2, pos2[:150], pos2[150:]); dt = time.perf_counter() - t0
assert r == Solution().getLastMoment(n2, pos2[:150], pos2[150:])
print(f"n=300 | mô phỏng: {r}  ({dt*1000:.1f} ms)")

n=1e4 | loop    : 9999  (0.69 ms)
n=1e4 | formula : 9999  (0.12 ms)
n=1e4 | order   : 9999  (12.51 ms)
n=1e4 | submit  : 9999  (0.10 ms)
n=300 | mô phỏng: 300  (26.1 ms)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `max(left)` khi `left` rỗng

Ví dụ 2 có `left = []` ⇒ `ValueError`. Ta bắt lỗi để minh hoạ.

In [17]:
def wrong1(n, left, right):
    return max(max(left), n - min(right))          # SAI khi một mảng rỗng

for n, l, r in [(7, [], [0, 1, 2]), (7, [3], [])]:
    try:
        print((n, l, r), "| sai:", wrong1(n, l, r))
    except ValueError as e:
        print((n, l, r), "| sai: ValueError ->", e)
    print("      đúng:", Solution().getLastMoment(n, l, r))

(7, [], [0, 1, 2]) | sai: ValueError -> max() iterable argument is empty
      đúng: 7
(7, [3], []) | sai: ValueError -> min() iterable argument is empty
      đúng: 3


**Sửa:** `max(left, default=0)`, `min(right, default=n)`.

### Lỗi 2 — Dùng `n − max(right)` thay vì `n − min(right)`

Con đi phải **tốn thời gian nhất** là con ở **gần đầu trái nhất** (vị trí nhỏ nhất), không phải lớn nhất.

In [18]:
def wrong2(n, left, right):
    return max(max(left, default=0), n - max(right, default=n))   # SAI

for (n, l, r), want in examples + [((10, [3], [2, 7]), 8)]:
    print((n, l, r), "| sai:", wrong2(n, l, r), "| đúng:", Solution().getLastMoment(n, l, r))

(4, [4, 3], [0, 1]) | sai: 4 | đúng: 4
(7, [], [0, 1, 2, 3, 4, 5, 6, 7]) | sai: 0 | đúng: 7
(7, [0, 1, 2, 3, 4, 5, 6, 7], []) | sai: 7 | đúng: 7
(10, [3], [2, 7]) | sai: 3 | đúng: 8


### Lỗi 3 — Nghĩ rằng con đi phải từ `p` mất `p` giây

Nhầm hướng: con đi phải phải đi tới đầu `n`, quãng đường `n − p`.

In [19]:
def wrong3(n, left, right):
    return max(left + right, default=0)              # SAI: con đi phải cũng tính p

for n, l, r in [(7, [], [0, 1, 2]), (10, [3, 9], [2, 7])]:
    print((n, l, r), "| sai:", wrong3(n, l, r), "| đúng:", Solution().getLastMoment(n, l, r))

(7, [], [0, 1, 2]) | sai: 2 | đúng: 7
(10, [3, 9], [2, 7]) | sai: 9 | đúng: 9


### Lỗi 4 — Mô phỏng với bước 1 giây và kiểm tra va chạm "cùng vị trí"

Hai con ở vị trí liền kề `p` và `p + 1` đi ngược chiều sẽ gặp nhau ở `p + 0.5` — bước 1 giây làm chúng **hoán đổi vị trí** mà không bao giờ "cùng vị trí" ⇒ mô phỏng bỏ sót va chạm. Đáp án (thời điểm cuối) tình cờ vẫn đúng nhờ tính chất bóng ma, nhưng **thời điểm rơi của từng con** thì sai.

In [20]:
def simulate_wrong_step(n, left, right):
    ants = [[p, -1, ("L", p)] for p in left] + [[p, 1, ("R", p)] for p in right]
    fall, t = {}, 0
    while ants:
        keep = []
        for a in ants:
            if (a[0] == 0 and a[1] == -1) or (a[0] == n and a[1] == 1):
                fall[a[2]] = t
            else:
                keep.append(a)
        ants = keep
        if not ants:
            break
        for a in ants:
            a[0] += a[1]                              # SAI: bước 1 giây, bỏ sót gặp ở x.5
        t += 1
        by = {}
        for a in ants:
            by.setdefault(a[0], []).append(a)
        for g in by.values():
            if len(g) == 2:
                g[0][1] *= -1; g[1][1] *= -1
    return fall

n, l, r = 4, [4, 3], [0, 1]
print("bước 1 giây (sai) :", dict(sorted(simulate_wrong_step(n, l, r).items())))
print("đúng (Cách 4)     :", dict(sorted(fall_times(n, l, r).items())))

bước 1 giây (sai) : {('L', 3): 3, ('L', 4): 4, ('R', 0): 4, ('R', 1): 3}
đúng (Cách 4)     : {('L', 3): 4, ('L', 4): 3, ('R', 0): 3, ('R', 1): 4}


**Sửa:** nhân đôi toạ độ (bước nửa giây), hoặc dùng lập luận bóng ma + giữ thứ tự.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Các vật **giống hệt nhau** va chạm rồi **bật ngược** với cùng tốc độ" ⇒ coi như **đi xuyên qua** nhau (đổi nhãn). Câu hỏi về **thời điểm/tập vị trí** thì va chạm không quan trọng.
- Câu hỏi về **một con cụ thể** ⇒ dùng thêm tính chất **giữ thứ tự**.
- Luôn để ý mảng có thể rỗng ⇒ `default=`.
- Mô phỏng chuyển động với gặp nhau ở nửa đơn vị ⇒ **nhân đôi toạ độ**.
- Bài "brainteaser" ⇒ viết mô phỏng nhỏ để kiểm chứng giả thuyết trước khi tin công thức.

## 15. Biến thể của bài

1. **Thời điểm con kiến đầu tiên rơi:** `min(min(left), n − max(right))` (Bài 4).
2. **Thời điểm rơi của một con cụ thể / con rơi ở đầu nào:** Cách 4 (giữ thứ tự).
3. **Đếm số lần va chạm:** mỗi cặp (con đi phải ở `p`, con đi trái ở `q`) với `p < q` gặp nhau **đúng một lần** trong thế giới bóng ma ⇒ số va chạm = số cặp như vậy (Bài 5).
4. **Kiến trên vòng tròn** (POJ "Ants" và các bài tương tự): không rơi, hỏi vị trí sau `T` giây ⇒ bóng ma + giữ thứ tự vòng.
5. **LC 2731 — Movement of Robots:** robot va chạm đổi hướng, hỏi tổng khoảng cách sau `d` giây ⇒ đúng mẹo bóng ma.

In [21]:
# Minh hoạ biến thể 5 (LC 2731): tổng khoảng cách đôi một sau d giây, robot va chạm đổi hướng
def sum_distance(nums, s, d):
    MOD = 10**9 + 7
    pos = sorted(x + d if c == "R" else x - d for x, c in zip(nums, s))
    total = pref = 0
    for i, p in enumerate(pos):
        total += i * p - pref
        pref += p
    return total % MOD

print(sum_distance([-2, 0, 2], "RLL", 3), sum_distance([1, 0], "RL", 2))

8 5


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng nửa giây | Nhân đôi toạ độ, xử lý va chạm | `O(n · A)` | `O(A)` | Kiểm chứng; hỏi chi tiết từng bước |
| 2. Vòng lặp bóng ma | Mỗi con đi thẳng: `p` hoặc `n − p` | `O(A)` | `O(1)` | Dễ giải thích |
| 3. Công thức | `max(max(left), n − min(right))` | `O(A)` | `O(1)` | Bản nộp gọn nhất |
| 4. Giữ thứ tự | Gán thời điểm rơi cho từng con thật | `O(A log A)` | `O(A)` | Hỏi về từng con kiến |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().getLastMoment(10, [3, 9], [2, 7])` bằng bao nhiêu?

### Bài 2 (tính tay)
Với Ví dụ 1 (`n = 4`, A ở 0 →, B ở 1 →, C ở 3 ←, D ở 4 ←): dùng tính chất giữ thứ tự, mỗi con rơi lúc nào và ở đầu nào?

### Bài 3 (điền TODO)
Hoàn thiện `last_side(n, left, right)`: trả `"left"`, `"right"` hoặc `"both"` — con kiến cuối cùng rơi ở đầu nào.

### Bài 4 (tự viết lại)
Viết `first_moment(n, left, right)`: thời điểm con kiến **đầu tiên** rơi khỏi ván.

### Bài 5 (mini challenge)
Viết `count_collisions(n, left, right)`: tổng số lần va chạm (mỗi lần hai con gặp nhau tính 1), trong `O(A log A)`. Kiểm chứng bằng mô phỏng.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def last_side(n, left, right):
    # TODO: so sánh max(left, default=-1) với n - min(right, default=n+1)
    pass

# Bài 4 — TODO
def first_moment(n, left, right):
    # TODO: con đi trái gần 0 nhất, con đi phải gần n nhất
    pass

# Bài 5 — TODO
def count_collisions(n, left, right):
    # TODO: đếm cặp (p trong right, q trong left) với p < q (sort + bisect)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đi trái: `max(3, 9) = 9`. (2) Đi phải: `10 − min(2, 7) = 8`. (3) Đáp án `9`.

**Bài 2.** (1) `L = 2` ⇒ hai con trái nhất (A, B) rơi đầu trái. (2) Thời điểm rơi đầu trái: `sorted(left) = [3, 4]` ⇒ A lúc 3, B lúc 4. (3) Đầu phải: `sorted(4 − 0, 4 − 1) = [3, 4]` ⇒ con phải nhất D lúc 3, C lúc 4.

**Bài 3.** (1) `a = max(left, default=-1)` (−1 nghĩa là không có). (2) `b = n − min(right)` nếu có, ngược lại −1. (3) So sánh `a` và `b`.

**Bài 4.** (1) Con đi trái rơi sớm nhất: `min(left)`. (2) Con đi phải rơi sớm nhất: `n − max(right)`. (3) Lấy `min` của hai giá trị có mặt.

**Bài 5.** (1) Trong thế giới bóng ma, bóng ma đi phải từ `p` và đi trái từ `q` gặp nhau đúng khi `p < q`. (2) Mỗi lần gặp của bóng ma ứng với đúng một va chạm thật. (3) Sort `left`, với mỗi `p` trong `right` đếm số `q > p` bằng `bisect_right`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
a1 = Solution().getLastMoment(10, [3, 9], [2, 7])
print(a1); assert a1 == 9

# Bài 2
ft = fall_times(4, [4, 3], [0, 1])
names = {("R", 0): "A", ("R", 1): "B", ("L", 3): "C", ("L", 4): "D"}
print({names[k]: v for k, v in ft.items()})
assert {names[k]: v for k, v in ft.items()} == {"A": 3, "B": 4, "D": 3, "C": 4}

9
{'A': 3, 'B': 4, 'D': 3, 'C': 4}


In [24]:
# Bài 3
def last_side_sol(n, left, right):
    a = max(left, default=-1)
    b = n - min(right) if right else -1
    return "both" if a == b else ("left" if a > b else "right")

# Bài 4
def first_moment_sol(n, left, right):
    cands = []
    if left:
        cands.append(min(left))
    if right:
        cands.append(n - max(right))
    return min(cands)

rng = random.Random(4)
for _ in range(800):
    n = rng.randint(1, 25)
    pos = rng.sample(range(n + 1), rng.randint(1, n + 1))
    left = [p for p in pos if rng.random() < 0.5]
    right = [p for p in pos if p not in left]
    _, fall, _, _ = simulate(n, left, right)
    t_first = min(fall.values()) // 2
    assert first_moment_sol(n, left, right) == t_first
    # bên rơi cuối cùng theo thế giới bóng ma: bóng ma đi trái rơi đầu trái, đi phải rơi đầu phải
    side = last_side_sol(n, left, right)
    last = Solution().getLastMoment(n, left, right)
    lefts_last = any(p == last for p in left)
    rights_last = any(n - p == last for p in right)
    assert side == ("both" if lefts_last and rights_last else ("left" if lefts_last else "right"))
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [25]:
# Bài 5
import bisect
def count_collisions_sol(n, left, right):
    L = sorted(left)
    return sum(len(L) - bisect.bisect_right(L, p) for p in right)

print(count_collisions_sol(4, [4, 3], [0, 1]), simulate(4, [4, 3], [0, 1])[2])
for _ in range(800):
    n = rng.randint(1, 25)
    pos = rng.sample(range(n + 1), rng.randint(1, n + 1))
    left = [p for p in pos if rng.random() < 0.5]
    right = [p for p in pos if p not in left]
    assert count_collisions_sol(n, left, right) == simulate(n, left, right)[2], (n, left, right)
print("Đáp án Bài 1–5: mọi assert đều qua")

4 4
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "va chạm = đi xuyên qua":**

```python
# mỗi vật coi như đi thẳng, không va chạm
last_left  = max(left, default=0)          # đi trái từ p: tới 0 sau p
last_right = n - min(right, default=n)     # đi phải từ p: tới n sau n - p
return max(last_left, last_right)
```

**Template mô phỏng (khi cần kiểm chứng):** nhân đôi toạ độ, mỗi bước: loại con ở đầu hướng ra ngoài → dịch → hai con trùng vị trí thì đổi hướng.

**Checklist trước khi submit:**
- [ ] Xử lý `left` hoặc `right` rỗng (`default=`).
- [ ] Con đi phải dùng `n − min(right)` (con **xa đầu phải nhất**).
- [ ] Con ở `0` đi trái / ở `n` đi phải rơi ngay lúc `t = 0`.
- [ ] Đã kiểm chứng mẹo bằng mô phỏng nhỏ.
- [ ] Đúng chữ ký: `getLastMoment(self, n: int, left: list[int], right: list[int]) -> int`.